# Publications markdown generator for academicpages

Takes a TSV of publications with metadata and converts them for use with [academicpages.github.io](academicpages.github.io). This is an interactive Jupyter notebook ([see more info here](http://jupyter-notebook-beginner-guide.readthedocs.io/en/latest/what_is_jupyter.html)). The core python code is also in `publications.py`. Run either from the `markdown_generator` folder after replacing `publications.tsv` with one containing your data.

TODO: Make this work with BibTex and other databases of citations, rather than Stuart's non-standard TSV format and citation style.


## Data format

The TSV needs to have the following columns: pub_date, title, venue, excerpt, citation, site_url, and paper_url, with a header at the top. 

- `excerpt` and `paper_url` can be blank, but the others must have values. 
- `pub_date` must be formatted as YYYY-MM-DD.
- `url_slug` will be the descriptive part of the .md file and the permalink URL for the page about the paper. The .md file will be `YYYY-MM-DD-[url_slug].md` and the permalink will be `https://[yourdomain]/publications/YYYY-MM-DD-[url_slug]`

This is how the raw file looks (it doesn't look pretty, use a spreadsheet or other program to edit and create).

## Import pandas

We are using the very handy pandas library for dataframes.

In [1]:
import pandas as pd

## Import TSV

Pandas makes this easy with the read_csv function. We are using a TSV, so we specify the separator as a tab, or `\t`.

I found it important to put this data in a tab-separated values format, because there are a lot of commas in this kind of data and comma-separated values can get messed up. However, you can modify the import statement, as pandas also has read_excel(), read_json(), and others.

In [117]:
df = pd.read_csv("database_csv/paper-20200422.csv", dtype=str)#.dropna()



In [137]:
df = df.fillna(' ')

## Escape special characters

YAML is very picky about how it takes a valid string, so we are replacing single and double quotes (and ampersands) with their HTML encoded equivilents. This makes them look not so readable in raw format, but they are parsed and rendered nicely.

In [138]:
html_escape_table = {
    "&": "&amp;",
    '"': "&quot;",
    "'": "&apos;"
    }

def html_escape(text):
    """Produce entities within text."""
    return "".join(html_escape_table.get(c,c) for c in text)

## Creating the markdown files

This is where the heavy lifting is done. This loops through all the rows in the TSV dataframe, then starts to concatentate a big string (```md```) that contains the markdown for each type. It does the YAML metadata first, then does the description for the individual page.

In [139]:
template = \
'''---
title: "{title}"
collection: papers
permalink: /papers/{file_name}
excerpt: '{excerpt}'
date: {pub_date}
venue: {venue}
paperurl: 'http://phanyoung.github.io/files/{file_name}.pdf'
citation: '{AUTHOR}."{title}[{type}]."<i>{venue}</i>, {year}, {volumn}:p.{pages}.'
---

摘要
-----
{excerpt}   
  

正文
-----
* [在线阅读]({paper_url})
* [论文下载](http://phanyoung.github.io/files/{file_name}.pdf)

Cite it: '{AUTHOR}."{title}[{type}]."<i>venue</i>, {year}, {volumn}:p.{pages}.'
'''

In [140]:
k_pattern = re.compile(r'{(.*?)}')

In [141]:
content_keys = k_pattern.findall(template)

In [142]:
def dict_strip(d):
    return {k: d[k].strip() for k in d.keys()}


def gen_md(d, bold_name_list = ['Fan Yang', '杨帆']):
    d = dict_strip(d)
    d['AUTHOR'] = d['author']
    for bold_name in bold_name_list:
        d['AUTHOR'] = d['AUTHOR'].replace(bold_name, '<b>'+bold_name+'</b>')
    content = {k: d[k] for k in content_keys}
    return template.format(**content), d['file_name']

These files are in the publications directory, one directory below where we're working from.

In [143]:
def gen_md_files(df, path=''):
    for _, d in df.iterrows():
        md_str, filename = gen_md(d)
        with open(path+filename+'.md', 'w') as f:
            f.write(md_str)


In [144]:
gen_md_files(df, '../_papers/')

In [145]:
m

['title',
 'file',
 'abstract',
 'date',
 'venue',
 'file_name',
 'AUTHOR',
 'title',
 'year',
 'volum',
 'pages',
 'SN',
 'date',
 'AUTHOR',
 'excerpt',
 'abstract',
 'online_url',
 'file_name',
 'AUTHOR',
 'title',
 'year',
 'volum',
 'pages']